<a href="https://colab.research.google.com/github/lillian181/INST750LG/blob/main/pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# SETUP: installs, imports, API client, and data loading
# ============================================================

!pip install -q openai pandas scikit-learn seaborn matplotlib tqdm

import os
import time
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from google.colab import userdata
from openai import OpenAI

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.model_selection import train_test_split

# ----------------------------
# Reproducibility
# ----------------------------
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# ----------------------------
# OpenAI setup
# ----------------------------
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
client = OpenAI()

# Use the same model for translation and classification so the comparison is consistent.

MODEL_NAME = "gpt-4o"

# ----------------------------
# Load data
# ----------------------------
df = pd.read_csv("dev.csv", sep="\t", header=None)
df.columns = ["label", "text"]

# Basic cleaning
df = df.dropna(subset=["label", "text"]).copy()
df["label"] = df["label"].astype(str).str.strip()
df["text"] = df["text"].astype(str).str.strip()

class_names = sorted(df["label"].unique().tolist())

print("Dataset shape:", df.shape)
print("Number of labels:", len(class_names))
print("Labels:", class_names)
print("\nLabel distribution:")
display(df["label"].value_counts())

In [ ]:

EVAL_SIZE = 300  # sample size chosen

if len(df) <= EVAL_SIZE:
    eval_df = df.copy()
else:
    # Stratified sampling requires each class to have at least 2 rows.
    label_counts = df["label"].value_counts()
    can_stratify = label_counts.min() >= 2

    if can_stratify:
        _, eval_df = train_test_split(
            df,
            test_size=EVAL_SIZE,
            stratify=df["label"],
            random_state=RANDOM_STATE
        )
    else:
        eval_df = df.sample(n=EVAL_SIZE, random_state=RANDOM_STATE)

eval_df = eval_df.reset_index(drop=True)

print("Evaluation sample shape:", eval_df.shape)
print("\nEvaluation sample label distribution:")
display(eval_df["label"].value_counts())

In [ ]:
# ============================================================
# STEP 2: Helper functions for translation, classification, and evaluation
# ============================================================

def call_openai_with_retry(messages, model=MODEL_NAME, temperature=0, max_retries=3, sleep_seconds=3):
    """
    Calls the OpenAI chat model with simple retry logic.
    """
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=temperature
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"OpenAI call failed after {max_retries} attempts: {e}")
                return "Error"
            time.sleep(sleep_seconds)


def translate_to_english(text):
    """
    Translates text to English while preserving the original meaning.
    If the text is already English, the model should return the original English text.
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are a careful translator. Translate the user's text into natural English. "
                "Preserve the original meaning, tone, named entities, and sentiment. "
                "Do not explain anything. Output only the English translation."
            )
        },
        {"role": "user", "content": text}
    ]
    return call_openai_with_retry(messages)


def classify_text(text, class_names):
    """
    Classifies the text into exactly one of the allowed labels.
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are a precise text classifier. "
                f"Classify the user's text into exactly one of these labels: {', '.join(class_names)}. "
                "Output only the label and nothing else."
            )
        },
        {"role": "user", "content": text}
    ]

    prediction = call_openai_with_retry(messages)
    prediction = prediction.strip()

    # Strict validation to prevent new/hallucinated labels.
    return prediction if prediction in class_names else "Error"


def evaluate_predictions(results_df, prediction_col, title):
    """
    Prints classification metrics and plots a confusion matrix.
    """
    clean = results_df[results_df[prediction_col] != "Error"].copy()

    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)
    print(f"Rows evaluated successfully: {len(clean)} / {len(results_df)}")
    print(f"Rows with API/model errors: {(results_df[prediction_col] == 'Error').sum()}")

    if len(clean) == 0:
        print("No successful predictions to evaluate.")
        return None

    y_true = clean["label"]
    y_pred = clean[prediction_col]

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, labels=class_names, average="macro", zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, labels=class_names, average="weighted", zero_division=0)

    print(f"Accuracy: {acc:.4f}")
    print(f"Macro F1: {macro_f1:.4f}")
    print(f"Weighted F1: {weighted_f1:.4f}")

    print("\nClassification report:")
    print(classification_report(
        y_true,
        y_pred,
        labels=class_names,
        target_names=class_names,
        zero_division=0
    ))

    cm = confusion_matrix(y_true, y_pred, labels=class_names)

    plt.figure(figsize=(10, 7))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(cmap="Blues", values_format="d", xticks_rotation=45)
    plt.title(title)
    plt.show()

    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "n_success": len(clean),
        "n_error": int((results_df[prediction_col] == "Error").sum())
    }


def bootstrap_metric_ci(y_true, y_pred, metric_func, n_bootstrap=1000, alpha=0.05):
    """
    Bootstrap confidence interval for a metric.
    This gives a better sense of metric stability than one score alone.
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    n = len(y_true)

    if n == 0:
        return np.nan, np.nan, np.nan

    scores = []

    for _ in range(n_bootstrap):
        idx = np.random.choice(np.arange(n), size=n, replace=True)
        score = metric_func(y_true[idx], y_pred[idx])
        scores.append(score)

    lower = np.percentile(scores, 100 * alpha / 2)
    upper = np.percentile(scores, 100 * (1 - alpha / 2))
    mean_score = np.mean(scores)

    return mean_score, lower, upper


def print_bootstrap_intervals(results_df, prediction_col, title):
    """
    Reports 95% bootstrap confidence intervals for accuracy and macro F1.
    """
    clean = results_df[results_df[prediction_col] != "Error"].copy()

    print("\n" + "=" * 70)
    print(f"Bootstrap 95% confidence intervals: {title}")
    print("=" * 70)

    if len(clean) == 0:
        print("No successful predictions to evaluate.")
        return

    y_true = clean["label"].values
    y_pred = clean[prediction_col].values

    acc_mean, acc_low, acc_high = bootstrap_metric_ci(
        y_true,
        y_pred,
        lambda yt, yp: accuracy_score(yt, yp)
    )

    f1_mean, f1_low, f1_high = bootstrap_metric_ci(
        y_true,
        y_pred,
        lambda yt, yp: f1_score(yt, yp, labels=class_names, average="macro", zero_division=0)
    )

    print(f"Accuracy CI: mean={acc_mean:.4f}, 95% CI=({acc_low:.4f}, {acc_high:.4f})")
    print(f"Macro F1 CI: mean={f1_mean:.4f}, 95% CI=({f1_low:.4f}, {f1_high:.4f})")

In [ ]:
# ============================================================
# STEP 3: Translate the evaluation sample to English
# ============================================================

translated_df = eval_df.copy()

tqdm.pandas()
translated_df["text_english"] = translated_df["text"].progress_apply(translate_to_english)

print("Sample translations:")
display(translated_df[["label", "text", "text_english"]].head(10))

# Save translations so you do not need to pay for translation again if runtime restarts.
translated_df.to_csv("translated_eval_sample.csv", index=False)
print("Saved: translated_eval_sample.csv")

In [ ]:
# ============================================================
# STEP 4: Evaluate the original, untranslated text
# ============================================================

results_df = translated_df.copy()

results_df["prediction_original"] = results_df["text"].progress_apply(
    lambda x: classify_text(x, class_names)
)

original_metrics = evaluate_predictions(
    results_df,
    prediction_col="prediction_original",
    title="OpenAI Zero-Shot Classification on Original Text"
)

print_bootstrap_intervals(
    results_df,
    prediction_col="prediction_original",
    title="Original Text"
)

In [ ]:
# ============================================================
# STEP 5: Evaluate the English-translated text
# ============================================================


results_df["prediction_english"] = results_df["text_english"].progress_apply(
    lambda x: classify_text(x, class_names)
)

english_metrics = evaluate_predictions(
    results_df,
    prediction_col="prediction_english",
    title="OpenAI Zero-Shot Classification on English-Translated Text"
)

print_bootstrap_intervals(
    results_df,
    prediction_col="prediction_english",
    title="English-Translated Text"
)

In [ ]:
# ============================================================
# STEP 6: Compare original vs English-translated evaluation
# ============================================================

if 'results_df' not in locals() or 'results_df' not in globals() or not isinstance(results_df, pd.DataFrame) or results_df.shape[0] != translated_df.shape[0] or \
   "prediction_original" not in results_df.columns or "prediction_english" not in results_df.columns:
    print("Re-initializing results_df and ensuring prediction columns are present.")
    results_df = translated_df.copy()

    if "prediction_original" not in results_df.columns:
        print("Calculating 'prediction_original' column.")
        results_df["prediction_original"] = results_df["text"].progress_apply(
            lambda x: classify_text(x, class_names)
        )

    if "prediction_english" not in results_df.columns:
        print("Calculating 'prediction_english' column.")
        results_df["prediction_english"] = results_df["text_english"].progress_apply(
            lambda x: classify_text(x, class_names)
        )

# Re-calculate original_metrics and english_metrics if they are not in scope,
# or if prediction columns were just calculated (to ensure fresh metrics).
if 'original_metrics' not in locals() or 'english_metrics' not in locals() or \
   "prediction_original" not in results_df.columns or "prediction_english" not in results_df.columns: # Re-check column presence for robustness
    print("Re-evaluating performance metrics.")
    original_metrics = evaluate_predictions(
        results_df,
        prediction_col="prediction_original",
        title="OpenAI Zero-Shot Classification on Original Text"
    )
    english_metrics = evaluate_predictions(
        results_df,
        prediction_col="prediction_english",
        title="OpenAI Zero-Shot Classification on English-Translated Text"
    )

comparison = pd.DataFrame([
    {"version": "Original text", **original_metrics},
    {"version": "English-translated text", **english_metrics}
])

display(comparison)

# Where did translation change the model's decision?
changed_predictions = results_df[
    (results_df["prediction_original"] != "Error") &
    (results_df["prediction_english"] != "Error") &
    (results_df["prediction_original"] != results_df["prediction_english"])
].copy()

print(f"Number of rows where translation changed the prediction: {len(changed_predictions)}")

display(changed_predictions[
    ["label", "text", "text_english", "prediction_original", "prediction_english"]
].head(20))

# Save final results for later analysis.
results_df.to_csv("original_vs_english_translation_results.csv", index=False)
print("Saved: original_vs_english_translation_results.csv")